In [ ]:
import time
import math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

# hyperparameters
TRAIN_SIZE = 10000
epochs = 10
learning_rate = 1e-2
batch_size = 8

# use tf32 for matmuls
torch.set_float32_matmul_precision("high")

# load mnist from raw binary (flattened 28x28 = 784 floats per image)
X_train_np = np.fromfile("data/X_train.bin", dtype=np.float32).reshape(60000, 784)
y_train_np = np.fromfile("data/y_train.bin", dtype=np.int32)
X_test_np = np.fromfile("data/X_test.bin", dtype=np.float32).reshape(10000, 784)
y_test_np = np.fromfile("data/y_test.bin", dtype=np.int32)

# normalize with mnist train mean/std
mean, std = 0.1307, 0.3081
X_train_np = (X_train_np - mean) / std
X_test_np = (X_test_np - mean) / std

# to tensors on gpu, shaped (batch, channels, height, width)
train_data = torch.from_numpy(X_train_np[:TRAIN_SIZE].reshape(-1, 1, 28, 28)).to("mps")
train_labels = torch.from_numpy(y_train_np[:TRAIN_SIZE]).long().to("mps")
test_data = torch.from_numpy(X_test_np.reshape(-1, 1, 28, 28)).to("mps")
test_labels = torch.from_numpy(y_test_np).long().to("mps")

In [ ]:
class MLP(nn.Module):
  def __init__(self, in_features, hidden_features, num_classes):
    super(MLP, self).__init__()
    self.fc1 = nn.Linear(in_features, hidden_features)
    self.relu = nn.ReLU()
    self.fc2 = nn.Linear(hidden_features, num_classes)

  def forward(self, x):
    x = x.reshape(batch_size, 28 * 28)  # flatten image into a row
    x = self.fc1(x)
    x = self.relu(x)
    x = self.fc2(x)
    return x

In [ ]:
model = MLP(in_features=784, hidden_features=256, num_classes=10).to("mps")

# xavier-style uniform init, biases zeroed
with torch.no_grad():
  fan_in_fc1 = model.fc1.weight.size(1)
  scale_fc1 = (6.0 / fan_in_fc1) ** 0.5
  model.fc1.weight.uniform_(-scale_fc1, scale_fc1)
  model.fc1.bias.zero_()

  fan_in_fc2 = model.fc2.weight.size(1)
  scale_fc2 = (6.0 / fan_in_fc2) ** 0.5
  model.fc2.weight.uniform_(-scale_fc2, scale_fc2)
  model.fc2.bias.zero_()

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=learning_rate)

In [ ]:
def train_timed(model, criterion, optimizer, epoch, timing_stats, epoch_losses):
  model.train()
  epoch_loss = 0.0
  iters_per_epoch = math.ceil(TRAIN_SIZE / batch_size)

  for i in range(iters_per_epoch):
    # load batch
    data_start = time.time()
    data = train_data[i * batch_size : (i + 1) * batch_size]
    target = train_labels[i * batch_size : (i + 1) * batch_size]
    data_end = time.time()
    timing_stats['data_loading'] += data_end - data_start

    optimizer.zero_grad()

    # forward pass
    forward_start = time.time()
    outputs = model(data)
    forward_end = time.time()
    timing_stats['forward'] += forward_end - forward_start

    # loss
    loss_start = time.time()
    loss = criterion(outputs, target)
    epoch_loss += loss.item()
    loss_end = time.time()
    timing_stats['loss_computation'] += loss_end - loss_start

    # backward pass
    backward_start = time.time()
    loss.backward()
    backward_end = time.time()
    timing_stats['backward'] += backward_end - backward_start

    # weight update
    update_start = time.time()
    optimizer.step()
    optimizer.zero_grad()
    update_end = time.time()
    timing_stats['weight_updates'] += update_end - update_start

  epoch_losses.append(epoch_loss / iters_per_epoch)
  return epoch_loss / iters_per_epoch

In [ ]:
def evaluate(model, test_data, test_labels):
  # average per-batch accuracy over the test set
  model.eval()

  total_batch_accuracy = torch.tensor(0.0, device="mps")
  num_batches = 0

  # no grads needed for eval
  with torch.no_grad():
    for i in range(len(test_data)):
      data = test_data[i * batch_size : (i + 1) * batch_size]
      target = test_labels[i * batch_size : (i + 1) * batch_size]

      # predicted class = argmax over logits
      outputs = model(data)
      _, predicted = torch.max(outputs.data, 1)

      correct_batch = (predicted == target).sum().item()
      total_batch = target.size(0)
      if total_batch != 0:
        batch_accuracy = correct_batch / total_batch
        total_batch_accuracy += batch_accuracy
        num_batches += 1

  avg_batch_accuracy = total_batch_accuracy / num_batches
  print(f"Average Batch Accuracy: {avg_batch_accuracy * 100:.2f}%")


if __name__ == "__main__":
  # per-stage timing accumulators
  timing_stats = {
    'data_loading': 0.0,
    'forward': 0.0,
    'loss_computation': 0.0,
    'backward': 0.0,
    'weight_updates': 0.0,
    'total_time': 0.0
  }
  epoch_losses = []

  total_start = time.time()

  # train
  for epoch in range(epochs):
    train_timed(model, criterion, optimizer, epoch, timing_stats, epoch_losses)
    print(f"Epoch {epoch} loss: {epoch_losses[epoch]:.4f}")

  total_end = time.time()
  timing_stats['total_time'] = total_end - total_start

  # timing breakdown
  print("\n=== pytorch mps timing breakdown ===")
  print(f"Total training time: {timing_stats['total_time']:.1f} seconds\n")

  print("Detailed Breakdown:")
  print(f"  Data loading:     {timing_stats['data_loading']:6.3f}s ({100.0 * timing_stats['data_loading'] / timing_stats['total_time']:5.1f}%)")
  print(f"  Forward pass:     {timing_stats['forward']:6.3f}s ({100.0 * timing_stats['forward'] / timing_stats['total_time']:5.1f}%)")
  print(f"  Loss computation: {timing_stats['loss_computation']:6.3f}s ({100.0 * timing_stats['loss_computation'] / timing_stats['total_time']:5.1f}%)")
  print(f"  Backward pass:    {timing_stats['backward']:6.3f}s ({100.0 * timing_stats['backward'] / timing_stats['total_time']:5.1f}%)")
  print(f"  Weight updates:   {timing_stats['weight_updates']:6.3f}s ({100.0 * timing_stats['weight_updates'] / timing_stats['total_time']:5.1f}%)")

  print("Finished Training")